# Modul 3 - Hand & Pose Tracking (IMV Laboratory) - Versi Colab

**Fokus:** cara kerja landmark dan tracking
**Hands-on:** virtual painter dan kontrol volume dengan jari
**Target:** peserta dapat melacak tangan dan membuat minimal satu interaksi yang dipicu gesture.

Notebook ini gabungan dari *Hands-on Landmark MediaPipe* (konsep) dan *Modul 3 Hand Tracking* (aplikasi).

> **Beda dengan versi lokal:** Colab tidak bisa membuka webcam lewat `cv2.VideoCapture` dan tidak punya `cv2.imshow`. Maka:
> - Konsep & tracking dasar memakai **satu foto** (kamera browser, upload, atau tangan sintetis).
> - Virtual Painter & Volume Control berjalan pada **video upload** atau **simulasi sintetis**, hasilnya ditampilkan sebagai cuplikan frame.
> - Untuk interaksi real-time dengan webcam, pakai versi lokal.

Pilih sumber input di sel setup (`INPUT_MODE`), default `"synthetic"` jadi notebook bisa dijalankan dari atas ke bawah tanpa izin apa pun.

## 0. Setup

In [ ]:
!pip install -q mediapipe

In [ ]:
import math
import os
import time
from types import SimpleNamespace

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np

plt.rcParams["figure.dpi"] = 100

# Urutan hasil fingers_up(): [jempol, telunjuk, tengah, manis, kelingking]
FINGER_NAMES = ["Jempol", "Telunjuk", "Tengah", "Manis", "Kelingking"]

# 21 landmark tangan: pasangan titik yang dihubungkan (rangka tangan)
CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),            # jempol
    (0, 5), (5, 6), (6, 7), (7, 8),            # telunjuk
    (5, 9), (9, 10), (10, 11), (11, 12),       # tengah
    (9, 13), (13, 14), (14, 15), (15, 16),     # manis
    (13, 17), (17, 18), (18, 19), (19, 20),    # kelingking
    (0, 17),                                   # sisi telapak
]


# ---------------------------------------------------------------- objek landmark
class LM:
    """Satu landmark ternormalisasi (x, y, z)."""
    def __init__(self, x, y, z=0.0):
        self.x, self.y, self.z = x, y, z


class FakeHand:
    """Meniru bentuk hand_landmarks MediaPipe (punya atribut .landmark)."""
    def __init__(self, lms):
        self.landmark = lms


# ---------------------------------------------------------------- detektor
class HandsCompat:
    """Pengganti mp.solutions.hands.Hands untuk mediapipe >= 0.10.30.

    Versi baru MediaPipe tidak lagi punya mp.solutions, jadi dipakai Tasks API
    (HandLandmarker). Model .task diunduh otomatis sekali saja.
    """
    MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
                 "hand_landmarker/float16/1/hand_landmarker.task")

    def __init__(self, max_num_hands=1, min_detection_confidence=0.5,
                 min_tracking_confidence=0.5):
        import urllib.request
        from mediapipe.tasks import python as mp_python
        from mediapipe.tasks.python import vision

        path = "hand_landmarker.task"
        if not os.path.exists(path):
            urllib.request.urlretrieve(self.MODEL_URL, path)
        options = vision.HandLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=path),
            running_mode=vision.RunningMode.IMAGE,
            num_hands=max_num_hands,
            min_hand_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self._lm = vision.HandLandmarker.create_from_options(options)

    def process(self, rgb):
        image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
        res = self._lm.detect(image)
        out = SimpleNamespace(multi_hand_landmarks=None, multi_handedness=None)
        if res.hand_landmarks:
            out.multi_hand_landmarks = [
                FakeHand([LM(p.x, p.y, p.z) for p in lms]) for lms in res.hand_landmarks
            ]
            out.multi_handedness = [
                SimpleNamespace(classification=[SimpleNamespace(label=h[0].category_name)])
                for h in res.handedness
            ]
        return out

    def close(self):
        self._lm.close()


def make_hands(max_hands=1, detection_conf=0.7, tracking_conf=0.6, static=False):
    """Buat detektor tangan. static=True untuk foto, False untuk video/webcam."""
    if hasattr(mp, "solutions"):   # mediapipe lama (<= 0.10.21)
        return mp.solutions.hands.Hands(
            static_image_mode=static,
            max_num_hands=max_hands,
            min_detection_confidence=detection_conf,
            min_tracking_confidence=tracking_conf,
        )
    return HandsCompat(max_hands, detection_conf, tracking_conf)  # mediapipe baru


# ---------------------------------------------------------------- fungsi bantu landmark
def to_pixels(hand_landmarks, w, h):
    """Landmark MediaPipe berskala 0-1 -> koordinat piksel (x, y)."""
    return [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks.landmark]


def fingers_up(hand_landmarks, label):
    """Return 5 boolean [jempol, telunjuk, tengah, manis, kelingking].

    Asumsi: frame sudah di-flip (mirror) dan telapak menghadap kamera.
    Jari selain jempol: ujung (tip) lebih tinggi dari sendi PIP -> terangkat.
    Jempol: dibandingkan di sumbu X, tergantung tangan kiri/kanan.
    """
    lm = hand_landmarks.landmark
    up = []
    if label == "Right":
        up.append(lm[4].x < lm[3].x)
    else:
        up.append(lm[4].x > lm[3].x)
    for tip in (8, 12, 16, 20):
        up.append(lm[tip].y < lm[tip - 2].y)
    return up


def analyze(hand_landmarks, label, w, h):
    """Landmark -> (titik piksel, status 5 jari)."""
    return to_pixels(hand_landmarks, w, h), fingers_up(hand_landmarks, label)


def detect(hands, frame_bgr):
    """Deteksi tangan pada satu frame BGR.

    Return list [(pts, up, label), ...] -- kosong jika tidak ada tangan.
    """
    h, w = frame_bgr.shape[:2]
    res = hands.process(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))  # BGR -> RGB
    info = []
    if res.multi_hand_landmarks:
        for lms, handed in zip(res.multi_hand_landmarks, res.multi_handedness):
            label = handed.classification[0].label                    # "Left" / "Right"
            pts, up = analyze(lms, label, w, h)
            info.append((pts, up, label))
    return info


def draw_skeleton(frame, pts, line_color=(0, 200, 255), dot_color=(255, 0, 0)):
    """Gambar rangka 21 titik pada frame (warna dalam urutan frame: BGR untuk frame OpenCV)."""
    for a, b in CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], line_color, 2, cv2.LINE_AA)
    for p in pts:
        cv2.circle(frame, p, 4, dot_color, -1, cv2.LINE_AA)


def show(images, titles, cols=3, size=(4, 4)):
    rows = math.ceil(len(images) / cols)
    plt.figure(figsize=(size[0] * cols, size[1] * rows))
    for i, (im, title) in enumerate(zip(images, titles)):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------------- tangan sintetis (fallback & simulasi)
# 21 landmark telapak terbuka dalam koordinat ternormalisasi
SYNTH_LMS = [
    (0.50, 0.92, 0.00),                                    # 0  wrist
    (0.42, 0.85, -0.01), (0.34, 0.78, -0.02),              # 1-2 thumb
    (0.28, 0.72, -0.03), (0.23, 0.66, -0.04),              # 3-4 thumb tip
    (0.40, 0.60, 0.00), (0.38, 0.45, -0.01),               # 5-6 index
    (0.37, 0.35, -0.02), (0.36, 0.26, -0.03),              # 7-8 index tip
    (0.50, 0.58, 0.00), (0.50, 0.40, -0.01),               # 9-10 middle
    (0.50, 0.29, -0.02), (0.50, 0.19, -0.03),              # 11-12 middle tip
    (0.60, 0.60, 0.00), (0.62, 0.43, -0.01),               # 13-14 ring
    (0.63, 0.32, -0.02), (0.64, 0.23, -0.03),              # 15-16 ring tip
    (0.69, 0.65, 0.00), (0.73, 0.52, -0.01),               # 17-18 pinky
    (0.75, 0.43, -0.02), (0.77, 0.36, -0.03),              # 19-20 pinky tip
]

# jari terangkat per pose: (jempol, telunjuk, tengah, manis, kelingking)
POSES = {
    "palm": (1, 1, 1, 1, 1),
    "fist": (0, 0, 0, 0, 0),
    "point": (0, 1, 0, 0, 0),
    "two": (0, 1, 1, 0, 0),
    "three": (0, 1, 1, 1, 0),
}
FINGER_CHAINS = {1: (5, 6, 7, 8), 2: (9, 10, 11, 12), 3: (13, 14, 15, 16), 4: (17, 18, 19, 20)}


def blank_frame(h=480, w=640):
    return np.full((h, w, 3), 235, np.uint8)


def make_synthetic_hand(h=480, w=640):
    """Kanvas berisi siluet telapak tangan (untuk fallback tanpa kamera/foto)."""
    canvas = blank_frame(h, w)
    pts = [(int(x * w), int(y * h)) for x, y, _ in SYNTH_LMS]
    skin = (150, 180, 215)  # BGR
    cv2.fillPoly(canvas, [np.array([pts[i] for i in (0, 1, 5, 9, 13, 17)])], skin, cv2.LINE_AA)
    for chain in ((1, 2, 3, 4), (5, 6, 7, 8), (9, 10, 11, 12), (13, 14, 15, 16), (17, 18, 19, 20)):
        cv2.polylines(canvas, [np.array([pts[i] for i in chain])], False, skin, 26, cv2.LINE_AA)
    return cv2.GaussianBlur(canvas, (9, 9), 0)


def synth_hand(pose="palm", index_tip=None, thumb_tip=None, middle_tip=None, scale=0.6):
    """Buat 21 landmark sintetis untuk sebuah pose.

    index_tip : (x, y) ternormalisasi tujuan ujung telunjuk (None = tangan di tengah)
    thumb_tip : (x, y) ternormalisasi tujuan ujung jempol (untuk simulasi cubitan)
    middle_tip: (x, y) ternormalisasi tujuan ujung jari tengah (untuk simulasi jari merapat)
    """
    lms = [list(p) for p in SYNTH_LMS]
    up = POSES[pose]
    for f, chain in FINGER_CHAINS.items():
        if not up[f]:                                   # jari menekuk: ujung di bawah sendi PIP
            pip_y = lms[chain[1]][1]
            lms[chain[2]][1] = pip_y + 0.04
            lms[chain[3]][1] = pip_y + 0.09
    if not up[0]:                                       # jempol menekuk ke dalam
        lms[4][0] = lms[3][0] + 0.06
    if index_tip is None:
        ox = oy = (1 - scale) / 2
    else:
        ox = index_tip[0] - lms[8][0] * scale
        oy = index_tip[1] - lms[8][1] * scale
    out = [LM(x * scale + ox, y * scale + oy, z) for x, y, z in lms]
    if thumb_tip is not None:
        out[4].x, out[4].y = thumb_tip
    if middle_tip is not None:
        out[12].x, out[12].y = middle_tip
    return out


def synth_info(pose="palm", w=640, h=480, **kw):
    """Pose sintetis -> format info yang sama dengan detect(): [(pts, up, label)]."""
    pts, up = analyze(FakeHand(synth_hand(pose, **kw)), "Right", w, h)
    return [(pts, up, "Right")]


def fallback_info(w, h):
    """Dipakai kalau tidak ada tangan terdeteksi: telapak terbuka sintetis (tanpa skala)."""
    lms = [LM(x, y, z) for x, y, z in SYNTH_LMS]
    pts, up = analyze(FakeHand(lms), "Right", w, h)
    return [(pts, up, "Right")]


# ---------------------------------------------------------------- tahap 'tracking dasar'
def tracking_step(frame, info, print_log=False):
    """Gambar rangka, ujung telunjuk, dan jumlah jari terangkat tiap tangan."""
    for pts, up, label in info:
        draw_skeleton(frame, pts)
        cv2.circle(frame, pts[8], 10, (0, 255, 0), cv2.FILLED)       # ujung telunjuk
        x0, y0 = pts[0]
        cv2.putText(frame, f"{label}: {sum(up)} jari", (x0 - 40, y0 + 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 120, 0), 2)
        if print_log:
            names = [n for n, u in zip(FINGER_NAMES, up) if u]
            print(f"{label} | terangkat: {names or '-'} | telunjuk (px): {pts[8]}")


# ---------------------------------------------------------------- Virtual Painter
class Painter:
    """Menggambar di udara.

    Telunjuk saja -> gambar | telunjuk + tengah -> pindah (pen diangkat)
    Telapak terbuka ditahan ~1 detik -> hapus semua.
    Tombol (mode live): 1-4 warna, e penghapus, c clear.
    """
    COLORS = {
        "1": ((255, 120, 0), "Biru"),
        "2": ((0, 200, 0), "Hijau"),
        "3": ((0, 0, 255), "Merah"),
        "4": ((0, 255, 255), "Kuning"),
    }
    CLEAR_HOLD_FRAMES = 25  # ~1 detik di 25-30 FPS

    def __init__(self):
        self.canvas = None
        self.prev_pt = None
        self.color, self.color_name = self.COLORS["1"]
        self.eraser = False
        self.clear_counter = 0

    def is_erasing(self, pts):
        """Hook: return True jika gesture penghapus aktif. Default: tidak pernah (ditimpa di studi kasus)."""
        return False

    def on_key(self, key):
        ch = chr(key)
        if ch == "c" and self.canvas is not None:
            self.canvas[:] = 0
        elif ch == "e":
            self.eraser = not self.eraser
        elif ch in self.COLORS:
            self.color, self.color_name = self.COLORS[ch]
            self.eraser = False

    def step(self, frame, info):
        if self.canvas is None or self.canvas.shape != frame.shape:
            self.canvas = np.zeros_like(frame)
        mode = "Tidak ada tangan"

        if info:
            pts, up, label = info[0]
            draw_skeleton(frame, pts)
            thumb, index, middle, ring, pinky = up
            tip = pts[8]

            if all(up):
                # Telapak terbuka -> tahan untuk clear
                mode = f"Hapus semua... {self.clear_counter * 100 // self.CLEAR_HOLD_FRAMES}%"
                self.clear_counter += 1
                self.prev_pt = None
                if self.clear_counter >= self.CLEAR_HOLD_FRAMES:
                    self.canvas[:] = 0
                    self.clear_counter = 0
            else:
                self.clear_counter = 0
                if index and middle and not ring and not pinky:
                    if self.is_erasing(pts):
                        mode = "Menghapus (gesture)"            # gesture penghapus
                        cv2.circle(frame, tip, 20, (0, 0, 0), 2)
                        if self.prev_pt is not None:
                            cv2.line(self.canvas, self.prev_pt, tip, (0, 0, 0), 40)
                        self.prev_pt = tip
                    else:
                        mode = "Pindah (pen diangkat)"
                        self.prev_pt = None
                        cv2.circle(frame, tip, 12, self.color, 2)
                elif index and not middle and not ring and not pinky:
                    mode = "Menggambar" if not self.eraser else "Menghapus"
                    cv2.circle(frame, tip, 8, self.color, cv2.FILLED)
                    if self.prev_pt is not None:
                        if self.eraser:
                            cv2.line(self.canvas, self.prev_pt, tip, (0, 0, 0), 40)
                        else:
                            cv2.line(self.canvas, self.prev_pt, tip, self.color, 8)
                    self.prev_pt = tip
                else:
                    mode = "Siap (angkat telunjuk)"
                    self.prev_pt = None
        else:
            self.prev_pt = None
            self.clear_counter = 0

        # Tempelkan canvas ke frame (piksel yang bukan hitam)
        mask = self.canvas.any(axis=2)
        frame[mask] = self.canvas[mask]

        cv2.putText(frame, f"Mode: {mode}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
        cv2.putText(frame, f"Warna: {'Penghapus' if self.eraser else self.color_name}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, self.color, 2)


# ---------------------------------------------------------------- Kontrol Volume
RATIO_MIN = 0.25   # jari menempel -> 0%   (kalibrasi kalau perlu)
RATIO_MAX = 1.40   # jari terbuka lebar -> 100%
SMOOTH = 0.2       # 0-1, makin kecil makin halus tapi makin lambat


class VolumeControl:
    """Jarak ujung jempol (4) - ujung telunjuk (8), dinormalisasi ukuran telapak (0 ke 5)."""

    def __init__(self, set_volume=None, smooth=SMOOTH):
        self.vol = 0.0
        self.set_volume = set_volume
        self.smooth = smooth

    def step(self, frame, info):
        h, w = frame.shape[:2]
        if info:
            pts = info[0][0]
            draw_skeleton(frame, pts)
            thumb_tip, index_tip = pts[4], pts[8]
            palm_size = math.dist(pts[0], pts[5])  # pergelangan -> pangkal telunjuk
            if palm_size > 0:
                ratio = math.dist(thumb_tip, index_tip) / palm_size
                target = float(np.interp(ratio, [RATIO_MIN, RATIO_MAX], [0, 100]))
                self.vol += (target - self.vol) * self.smooth      # smoothing (EMA)
                if self.set_volume:
                    self.set_volume(self.vol)
            cv2.line(frame, thumb_tip, index_tip, (255, 0, 255), 3)
            cv2.circle(frame, thumb_tip, 10, (255, 0, 255), cv2.FILLED)
            cv2.circle(frame, index_tip, 10, (255, 0, 255), cv2.FILLED)

        # Bar volume
        bar_top, bar_bottom = int(h * 0.2), int(h * 0.8)
        fill = int(np.interp(self.vol, [0, 100], [bar_bottom, bar_top]))
        cv2.rectangle(frame, (30, bar_top), (65, bar_bottom), (0, 0, 0), 2)
        cv2.rectangle(frame, (30, fill), (65, bar_bottom), (0, 200, 0), cv2.FILLED)
        cv2.putText(frame, f"{int(self.vol)}%", (20, bar_bottom + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 150, 0), 2)

In [ ]:
# Sumber foto untuk Bagian A & tracking dasar:
#   "synthetic" = tangan sintetis (tanpa izin apa pun)
#   "webcam"    = ambil foto dari kamera browser
#   "upload"    = upload foto sendiri
INPUT_MODE = "synthetic"

# Untuk Painter & Volume: isi dengan nama file video yang di-upload, mis. "demo.mp4".
# None = pakai simulasi sintetis. Cara upload:
#   from google.colab import files; up = files.upload(); VIDEO_PATH = next(iter(up))
VIDEO_PATH = None


def colab_take_photo(quality=0.9):
    """Ambil 1 foto dari webcam browser (standar Colab, via JavaScript)."""
    from base64 import b64decode
    from google.colab.output import eval_js
    from IPython.display import Javascript, display

    display(Javascript("""
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const btn = document.createElement('button');
      btn.textContent = 'Ambil foto';
      div.appendChild(btn);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => btn.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }"""))
    data = eval_js(f"takePhoto({quality})")
    arr = np.frombuffer(b64decode(data.split(",")[1]), np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


def load_frame(mode=INPUT_MODE):
    """Ambil 1 frame sesuai INPUT_MODE; gagal apa pun -> kanvas sintetis."""
    try:
        if mode == "webcam":
            return cv2.flip(colab_take_photo(), 1), "foto webcam browser"
        if mode == "upload":
            from google.colab import files
            up = files.upload()
            img = cv2.imdecode(np.frombuffer(next(iter(up.values())), np.uint8), cv2.IMREAD_COLOR)
            if img is not None:
                return cv2.flip(img, 1), "foto upload"
    except Exception as e:
        print(f"[!] Input '{mode}' gagal ({e}). Pakai tangan sintetis.")
    return make_synthetic_hand(), "kanvas sintetis 480x640"


def get_sequence(kind):
    """Urutan frame untuk Painter/Volume: video upload kalau ada, kalau tidak simulasi."""
    if VIDEO_PATH and os.path.exists(VIDEO_PATH):
        return sequence_from_video(VIDEO_PATH, make_hands(max_hands=1)), f"video: {VIDEO_PATH}"
    seq = synthetic_painter_sequence() if kind == "painter" else synthetic_volume_sequence()
    return seq, "simulasi sintetis"

Fungsi simulasi dan pembaca video (dipakai oleh bagian B):

In [ ]:
def synthetic_painter_sequence(w=640, h=480):
    """Simulasi: gambar gelombang -> angkat pen (2 jari) -> gambar garis lurus."""
    steps = []
    for t in np.linspace(0, 1, 30):                                   # menggambar gelombang
        steps.append(("point", (0.15 + 0.7 * t, 0.35 + 0.1 * math.sin(t * 3 * math.pi))))
    for t in np.linspace(0, 1, 8):                                    # pindah, pen diangkat
        steps.append(("two", (0.85 - 0.7 * t, 0.35 + 0.35 * t)))
    for t in np.linspace(0, 1, 20):                                   # garis lurus
        steps.append(("point", (0.15 + 0.7 * t, 0.7)))
    for pose, tip in steps:
        yield blank_frame(h, w), synth_info(pose, w, h, index_tip=tip)


def synthetic_volume_sequence(w=640, h=480):
    """Simulasi: jempol dan telunjuk membuka lalu menutup."""
    base = synth_hand("palm")
    i8 = np.array([base[8].x, base[8].y])
    t4 = np.array([base[4].x, base[4].y])
    for t in list(np.linspace(0, 1.12, 24)) + list(np.linspace(1.12, 0, 24)):
        thumb = tuple(i8 + t * (t4 - i8))                              # t=0 menempel, t~1.1 terbuka lebar
        yield blank_frame(h, w), synth_info("palm", w, h, thumb_tip=thumb)


def synthetic_eraser_sequence(w=640, h=480):
    """Simulasi studi kasus: garis lurus -> jari merapat menghapus bagian tengahnya."""
    for t in np.linspace(0, 1, 30):                                   # garis horizontal
        yield blank_frame(h, w), synth_info("point", w, h, index_tip=(0.15 + 0.7 * t, 0.5))
    for t in np.linspace(0, 1, 6):                                    # pindah ke awal area hapus
        yield blank_frame(h, w), synth_info("two", w, h, index_tip=(0.85 - 0.45 * t, 0.5))
    for t in np.linspace(0, 1, 14):                                   # jari 8 & 12 merapat, geser
        x = 0.40 + 0.2 * t
        yield blank_frame(h, w), synth_info("two", w, h, index_tip=(x, 0.5), middle_tip=(x + 0.03, 0.48))


def sequence_from_video(path, hands, max_frames=120, stride=2):
    """Baca file video -> (frame ter-mirror, info) per frame."""
    cap = cv2.VideoCapture(path)
    n = i = 0
    while n < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        i += 1
        if i % stride:
            continue
        n += 1
        frame = cv2.flip(frame, 1)
        yield frame, detect(hands, frame)
    cap.release()


def run_offline(seq, step, every=10, max_snap=6, cols=3, show_last=True):
    """Jalankan `step` pada urutan frame tanpa jendela, tampilkan beberapa cuplikan."""
    shots, titles, last = [], [], None
    for i, (frame, info) in enumerate(seq):
        step(frame, info)
        last = frame
        if i % every == 0 and len(shots) < max_snap:
            shots.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            titles.append(f"frame {i}")
    if show_last and last is not None:
        shots.append(cv2.cvtColor(last, cv2.COLOR_BGR2RGB))
        titles.append("hasil akhir")
    show(shots, titles, cols=cols, size=(4, 3))

## Bagian A - Konsep Landmark & Koordinat

**Dua model landmark MediaPipe:**
- **Hand Landmarks - 21 titik (indeks 0-20):** 1 pergelangan + 4 titik x 5 jari. Dipakai pada Virtual Painter & Volume Control.
- **Pose Landmarks - 33 titik (indeks 0-32):** kerangka tubuh penuh (hidung, mata, telinga, bahu, siku, pergelangan, pinggul, lutut, mata kaki). Dipakai untuk analisis postur. Cara kerja dan konsep koordinatnya sama dengan tangan.

**Alur tracking tiap frame:** baca frame -> mirror -> BGR ke RGB -> `hands.process()` -> ambil landmark -> gambar / hitung gesture.

> OpenCV membaca gambar dalam **BGR**, sedangkan MediaPipe & Matplotlib bekerja di **RGB**. Konversi `cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)` **wajib** sebelum `hands.process()` dan sebelum `plt.imshow`. Jika terlewat: deteksi gagal dan warna tampil tertukar.

### 1. Akuisisi Citra & Deteksi 21 Landmark

Sumber foto ditentukan oleh `INPUT_MODE`. Jika tidak ada tangan terdeteksi, dipakai **21 landmark sintetis** sehingga sel berikutnya bebas dari error `NoneType`.

In [ ]:
frame_bgr, source = load_frame()
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)      # wajib: BGR -> RGB
h, w, c = frame_rgb.shape

hands_img = make_hands(max_hands=1, detection_conf=0.5, static=True)
results = hands_img.process(frame_rgb)

if results.multi_hand_landmarks:
    hand_lms = list(results.multi_hand_landmarks[0].landmark)
    label = results.multi_handedness[0].classification[0].label
    source += " | landmark asli MediaPipe"
else:
    hand_lms = [LM(x, y, z) for x, y, z in SYNTH_LMS]
    label = "Right"
    source += " | FALLBACK landmark sintetis"

annotated = frame_rgb.copy()
pts_vis = [(int(lm.x * w), int(lm.y * h)) for lm in hand_lms]
draw_skeleton(annotated, pts_vis, line_color=(255, 200, 0), dot_color=(255, 0, 0))   # frame RGB

print("Sumber citra :", source)
print("Jumlah titik :", len(hand_lms), "| shape (h, w, c) =", (h, w, c), "| tangan:", label)
show([frame_rgb, annotated], ["Citra Asli (RGB)", "Visualisasi 21 Landmark + Koneksi"], cols=2, size=(5, 4))

In [ ]:
HAND_NAMES = [
    "WRIST",
    "THUMB_CMC", "THUMB_MCP", "THUMB_IP", "THUMB_TIP",
    "INDEX_MCP", "INDEX_PIP", "INDEX_DIP", "INDEX_TIP",
    "MIDDLE_MCP", "MIDDLE_PIP", "MIDDLE_DIP", "MIDDLE_TIP",
    "RING_MCP", "RING_PIP", "RING_DIP", "RING_TIP",
    "PINKY_MCP", "PINKY_PIP", "PINKY_DIP", "PINKY_TIP",
]
print(f"Hand landmarks: {len(HAND_NAMES)} titik | Pose landmarks: 33 titik")
for i in (0, 4, 8, 12, 16, 20):
    print(f"  ID {i:>2} = {HAND_NAMES[i]}")

**Indeks kunci tangan yang wajib dihafal:**
- `0` = Wrist (pergelangan) -> titik acuan kedalaman `z`
- `4` = Thumb Tip (ujung jempol) -> Volume Control
- `8` = Index Tip (ujung telunjuk) -> kursor Virtual Painter
- `12` = Middle Tip, `16` = Ring Tip, `20` = Pinky Tip -> finger counter

### 2. Sistem Koordinat Ternormalisasi (x, y, z)

- `x` = horizontal, `0.0` (tepi kiri) -> `1.0` (tepi kanan)
- `y` = vertikal, `0.0` (tepi atas) -> `1.0` (tepi bawah). Makin ke bawah, nilai `y` makin **besar**
- `z` = kedalaman relatif terhadap pergelangan; makin **negatif** berarti makin dekat ke kamera
- Ketiganya rasio float terhadap ukuran frame, **bukan** piksel.

In [ ]:
for idx in (0, 4, 8, 12, 16, 20):
    lm = hand_lms[idx]
    print(f"ID {idx:>2} | x = {lm.x:.4f}  y = {lm.y:.4f}  z = {lm.z:+.4f}")

xs = [lm.x for lm in hand_lms]
ys = [lm.y for lm in hand_lms]

plt.figure(figsize=(6, 7))
for a, b in CONNECTIONS:
    plt.plot([xs[a], xs[b]], [ys[a], ys[b]], color="#bbbbbb", lw=1.5, zorder=1)
plt.scatter(xs, ys, c=range(len(xs)), cmap="jet", s=70, zorder=2)
for i, (x, y) in enumerate(zip(xs, ys)):
    plt.annotate(str(i), (x, y), xytext=(8, -8), textcoords="offset points", fontsize=9)
plt.xlim(-0.05, 1.05)
plt.ylim(1.05, -0.05)                                       # sumbu y dibalik: 0 di atas
plt.gca().set_aspect("equal")
plt.grid(alpha=0.3)
plt.title("Peta Landmark pada Ruang Ternormalisasi (0.0 - 1.0)")
plt.xlabel("x  (0 = kiri, 1 = kanan)")
plt.ylabel("y  (0 = atas, 1 = bawah)")
plt.show()

### 3. Konversi Koordinat ke Piksel Layar (cx, cy)

`cv2.circle`, `cv2.line`, dan `cv2.putText` hanya menerima **koordinat integer piksel**. Tanpa konversi, titik tergambar di posisi salah (sering menumpuk di pojok kiri atas).

- `cx = int(lm.x * w)` -> kolom piksel
- `cy = int(lm.y * h)` -> baris piksel
- Dimensi diambil dengan `h, w, c = frame.shape` (urutan OpenCV: tinggi, lebar, channel).
- Simpan hasilnya ke `lm_list = [[id, cx, cy], ...]` agar mudah difilter per-ID.

In [ ]:
lm_list = []
for idx, lm in enumerate(hand_lms):
    cx, cy = int(lm.x * w), int(lm.y * h)
    lm_list.append([idx, cx, cy])

print("5 titik pertama [id, cx, cy]:")
for row in lm_list[:5]:
    print("  ", row)

canvas_ids = frame_rgb.copy()
for idx, cx, cy in lm_list:
    cv2.circle(canvas_ids, (cx, cy), 5, (255, 0, 0), -1, cv2.LINE_AA)
    cv2.putText(canvas_ids, str(idx), (cx + 7, cy - 7), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (0, 150, 0), 1, cv2.LINE_AA)

show([canvas_ids], ["Pemetaan 21 Landmark ke Piksel (cx, cy)"], cols=1, size=(8, 5))

### 4. Isolasi Landmark untuk Proyek

Kita tidak perlu semua 21 titik, cukup filter ID tertentu.

**Virtual Painter (kursor kuas):** kursor = `(cx, cy)` dari **ID 8**. Telunjuk dianggap **terangkat** jika `cy` ID 8 **<** `cy` ID 6 (ingat: `cy` kecil = posisi lebih tinggi di layar).

**Volume Control (jarak cubitan):** ambil **ID 4** dan **ID 8**, hitung jarak Euclidean `math.hypot(x2 - x1, y2 - y1)`. Makin lebar cubitan, makin besar nilainya.

In [ ]:
base = frame_rgb.copy()

# --- 1) Simulasi Virtual Painter: ID 8 sebagai kursor kuas ---
painter_view = base.copy()
p8 = (lm_list[8][1], lm_list[8][2])
cv2.line(painter_view, (p8[0] - 150, p8[1] + 70), p8, (255, 0, 255), 8, cv2.LINE_AA)   # simulasi goresan
cv2.circle(painter_view, p8, 16, (255, 0, 255), -1, cv2.LINE_AA)
cv2.circle(painter_view, p8, 16, (255, 255, 255), 2, cv2.LINE_AA)
cv2.putText(painter_view, "ID 8 = kursor kuas", (10, 25), cv2.FONT_HERSHEY_SIMPLEX,
            0.6, (0, 0, 0), 2, cv2.LINE_AA)

# --- 2) Simulasi Volume Control: jarak ID 4 <-> ID 8 ---
volume_view = base.copy()
p4 = (lm_list[4][1], lm_list[4][2])
cv2.line(volume_view, p4, p8, (255, 255, 0), 3, cv2.LINE_AA)
cv2.circle(volume_view, p4, 9, (0, 255, 0), -1, cv2.LINE_AA)     # jempol
cv2.circle(volume_view, p8, 9, (255, 0, 0), -1, cv2.LINE_AA)     # telunjuk
mid = ((p4[0] + p8[0]) // 2, (p4[1] + p8[1]) // 2)
cv2.circle(volume_view, mid, 7, (255, 255, 255), -1, cv2.LINE_AA)
cv2.circle(volume_view, mid, 7, (0, 0, 0), 2, cv2.LINE_AA)
dist_px = math.hypot(p8[0] - p4[0], p8[1] - p4[1])
cv2.putText(volume_view, f"jarak ID4-ID8 = {dist_px:.1f} px", (10, 25), cv2.FONT_HERSHEY_SIMPLEX,
            0.6, (0, 0, 0), 2, cv2.LINE_AA)

print(f"ID 4 (jempol)   -> (cx, cy) = {p4}")
print(f"ID 8 (telunjuk) -> (cx, cy) = {p8}")
print(f"Jarak Euclidean ID 4 - ID 8 = {dist_px:.2f} px")
print("Telunjuk terangkat?", lm_list[8][2] < lm_list[6][2])

show([frame_rgb, painter_view, volume_view],
     ["Frame Asli", "Simulasi Virtual Painter", "Simulasi Volume Control"],
     cols=3, size=(5, 4))

## Bagian B - Hands-on

Mulai dari sini semua logika dibungkus fungsi/kelas (`tracking_step`, `Painter`, `VolumeControl`) yang menerima `info = [(pts, up, label), ...]`, yaitu titik piksel, status 5 jari `[jempol, telunjuk, tengah, manis, kelingking]`, dan label tangan.

### 5. Hand Tracking Dasar

MediaPipe Hands mendeteksi **21 landmark** per tangan. Jari selain jempol dianggap terangkat jika ujung (tip) lebih tinggi dari sendi PIP (`lm[tip].y < lm[tip-2].y`). Jempol dibandingkan di sumbu X sesuai tangan kiri/kanan.

In [ ]:
hands5 = make_hands(max_hands=5, detection_conf=0.5, static=True)
info = detect(hands5, frame_bgr) or fallback_info(w, h)   # tidak terdeteksi -> landmark sintetis

view = frame_bgr.copy()
tracking_step(view, info, print_log=True)
show([cv2.cvtColor(view, cv2.COLOR_BGR2RGB)], ["Hand tracking + hitung jari"], cols=1, size=(7, 5))

### 6. Virtual Painter

Menggambar di udara dengan jari telunjuk.

| Gesture | Aksi |
|---|---|
| Telunjuk saja terangkat | **Menggambar** |
| Telunjuk + tengah terangkat | Pindah posisi (pen diangkat) |
| Telapak terbuka (5 jari) ditahan ~1 detik | **Hapus semua** |

Di Colab, gesture diproses pada urutan frame (video upload atau simulasi: gambar gelombang -> angkat pen -> gambar garis lurus). Perhatikan jeda antara dua goresan saat mode **Pindah**.

In [ ]:
seq, src = get_sequence("painter")
print("Sumber:", src)
painter = Painter()
run_offline(seq, painter.step, every=10, max_snap=5, cols=3)

### 7. Kontrol Volume dengan Jari

Jarak ujung jempol (4) ke ujung telunjuk (8) dipetakan ke volume 0-100%. Jarak **dinormalisasi** dengan ukuran telapak (landmark 0 ke 5), jadi hasilnya konsisten walau tangan dekat atau jauh dari kamera. Nilai dihaluskan dengan smoothing (EMA) supaya bar tidak goyang.

Simulasi: jempol dan telunjuk membuka lalu menutup, bar volume mengikuti jarak. Di Colab `SMOOTH` dinaikkan ke 0.5 agar bar cepat mengejar pada urutan yang pendek.

In [ ]:
seq, src = get_sequence("volume")
print("Sumber:", src)
vc = VolumeControl(smooth=0.5)
run_offline(seq, vc.step, every=6, max_snap=8, cols=4, show_last=False)

### 8. Studi Kasus: Penghapus dengan Gesture

**Masalah:** di Virtual Painter, menghapus harus menekan tombol `e` di keyboard, padahal tangan sedang "di udara". Rancang dan kodekan **sendiri dari nol** sebuah painter mini yang bisa menghapus hanya dengan gesture.

**Syarat minimal:**
1. Telunjuk saja terangkat -> menggambar garis (warna bebas).
2. Telunjuk + jari tengah terangkat dan **merenggang** -> pen diangkat (tidak menggambar, tidak menghapus).
3. Telunjuk + jari tengah terangkat dan **merapat** -> menghapus coretan di bawah ujung jari.
4. Coretan harus tetap ada di antar-frame (butuh *kanvas* yang bertahan).

**Kontrak fungsi:** tulis `my_step(frame, info)`. Tiap frame fungsi ini dipanggil dengan:
- `frame`: gambar BGR (ubah langsung dengan fungsi `cv2`)
- `info`: list tangan `[(pts, up, label), ...]`, kosong jika tidak ada tangan. `pts` = 21 titik piksel, `up` = `[jempol, telunjuk, tengah, manis, kelingking]` (True/False), `label` = "Left"/"Right"

**Alat yang sudah tersedia:** `draw_skeleton(frame, pts)`, `math.dist(a, b)`, `cv2.line`, `cv2.circle`, `cv2.putText`, `np.zeros_like`.

**Cara menguji:** buat sel kode baru di bawah bagian ini, tulis `my_step` di sana, lalu jalankan:
- Simulasi: `run_offline(synthetic_eraser_sequence(), my_step, every=15, max_snap=3, cols=4)`. Hasil yang benar = garis lurus yang bagian tengahnya hilang.
- Video sendiri (opsional): isi `VIDEO_PATH` di sel setup, lalu pakai `get_sequence("painter")` sebagai pengganti `synthetic_eraser_sequence()`.

<details>
<summary>Petunjuk (buka kalau buntu)</summary>

- Keadaan antar-frame (kanvas, titik sebelumnya) bisa disimpan di variabel global (`global canvas, prev_pt`) atau di dalam sebuah kelas.
- Gambar di kanvas terpisah, lalu tempel ke `frame` di akhir: `mask = canvas.any(axis=2); frame[mask] = canvas[mask]`.
- Menghapus = menggambar garis **hitam** tebal di kanvas.
- Titik sebelumnya (`prev_pt`) harus di-reset ke `None` saat pen diangkat, supaya tidak muncul garis penghubung.
- Jarak jari merapat: `math.dist(pts[8], pts[12])`. Mulai dengan ambang sekitar 40 piksel, lalu kalibrasi.
</details>

**Bahan diskusi setelah jalan:**
1. Kenapa `fingers_up()` saja tidak cukup untuk membedakan merapat vs merenggang?
2. Apa efek ambang terlalu besar / terlalu kecil?
3. Kalau tangan menjauh dari kamera, jarak piksel mengecil. Bagaimana menormalisasinya? (petunjuk: bagian 7)

## 9. Tugas: Smart Room Gesture Controller

> Dikerjakan **setelah sesi** (tenggat menyusul). Kerjakan dari nol di sel kode baru.

**Skenario:** mengontrol lampu dan kipas tanpa sentuh. Jumlah jari terangkat = perintah.

| Jari terangkat | Perintah |
|---|---|
| 0 (kepalan) | SEMUA OFF |
| 1 | LAMPU 1 ON |
| 2 | LAMPU 2 ON |
| 3 | KIPAS ON |
| 5 (telapak) | SEMUA ON |
| lainnya | `-` (abaikan) |

**Yang harus dikerjakan:**
1. **Wajib:** tulis `gesture_command(up)`. Inputnya `up = [jempol, telunjuk, tengah, manis, kelingking]` (True/False), outputnya teks perintah sesuai tabel.
2. **Wajib:** tulis `controller_step(frame, info)` yang menampilkan perintah di pojok kiri atas frame (`cv2.putText`) dan menggambar rangka tangan (`draw_skeleton`). Format `info` sama seperti studi kasus.
3. **Wajib:** *debounce*. Perintah resmi berubah hanya jika gesture yang sama stabil >= 10 frame berturut-turut, supaya teks tidak berkedip.
4. **Bonus:** dimmer lampu memakai jarak jempol-telunjuk (rumus dari bagian 7), tampilkan sebagai bar 0-100%.

**Cara menguji:**
- Untuk tiap pose di `["fist", "point", "two", "three", "palm"]`, panggil `controller_step(blank_frame(), synth_info(pose))`, lalu tampilkan hasilnya dengan `show(...)`. Perintah yang muncul harus sesuai tabel.
- Debounce diuji dengan memanggil `controller_step` berulang pada pose yang sama (mis. dalam loop 15 kali) dan memeriksa kapan perintah berubah.

**Yang dikumpulkan:** notebook yang sudah diisi + tangkapan layar hasil uji. Di proyek nyata, perintah ini dikirim ke ESP32 lewat MQTT atau serial.

## Troubleshooting

- **Error numpy / ABI setelah `pip install`:** *Runtime -> Restart session*, lalu jalankan ulang dari sel setup.
- **Kamera tidak mau menyala:** izinkan akses kamera di browser (ikon gembok di address bar), lalu jalankan ulang sel.
- **Jempol terbaca terbalik:** foto/video dari kamera di-mirror otomatis oleh `load_frame()` / `sequence_from_video()`. Pastikan telapak menghadap kamera.
- **Ingin versi live (video real-time dengan webcam):** pakai notebook versi **lokal**.